# 01 · Pipeline de extracción: FASTA + CSV → MongoDB

**Entregable 2** · Bases de Datos II · PanTEon Database v1.6.1

Rama: `feature/pipeline-limpieza`

---

## Alcance (guía, §4.2)

- [ ] Leer y validar el FASTA: separar `ID`, `CLASE/ORDEN/SUPERFAMILIA` y `@Especie`; verificar caracteres válidos, presencia de `N` y longitudes.
- [ ] Leer y limpiar el CSV: conservar **solo** las 9 columnas (`Kingdom`, `Phylum`, `Class`, `Order`, `Family`, `Species`, `Source`, `Source 2`, `Notes`) y descartar `Library?`, `Selected`, `TE_count`, `MCH_Parameters`.
- [ ] Normalizar los nombres de especie en ambos lados y cruzar por `Species`.
- [ ] Reportar huérfanos en **ambos** sentidos (FASTA sin CSV y CSV sin FASTA).
- [ ] Calcular al menos dos atributos derivados por secuencia: longitud, %GC, %N.
- [ ] Calcular los *k*-mers de `k=1` a `k=6` y **guardarlos en MongoDB**, definiendo y documentando la política para ventanas con `N` u otros caracteres fuera de `{A, C, G, T}`.
- [ ] Cargar con `insert_many` o `bulk_write` (`pymongo`).
- [ ] Dejar el notebook ejecutable de principio a fin (requisito de reproducibilidad).

## Reglas de trabajo

1. Validar el diseño con un subconjunto de ~5 especies antes de escalar a las ~240.000 secuencias.
2. No subir datos crudos: `data_raw/` está en `.gitignore`.
3. Cada decisión de diseño se anota en `docs/decisiones.md` **en el momento**, no al final.


In [1]:
import sys, os
print(sys.executable)
print(os.getcwd())
print(os.listdir("../data_raw"))

C:\Users\camil\.conda\envs\transponibles\python.exe
C:\Users\camil\Documents\BDProyectoCorteII\panteon-mongodb-bd2\notebooks
['PanTEon_Database_metadata_v1.6.1.csv', 'PanTEon_Database_v1.6.1.fasta']


In [5]:
import pandas as pd

df = pd.read_csv("../data_raw/PanTEon_Database_metadata_v1.6.1.csv")
df_prueba = df.sample(9, random_state=1)
df_prueba

,Kingdom,Phylum,Class,Order,Family,Species,Library?,Source,Source 2,Notes,Selected,TE_count,MCH_Parameters
231,Fungi,Basidiomycota,Agaricomycetes,Agaricales,Physalacriaceae,Armillaria cepistipes,Complete,RM2+MCH,NaN,"ENSEMBL ""2023""",RM2+MCH,28,"x=10, e=500, c=1"
1443,Fungi,Basidiomycota,Agaricomycetes,Russulales,Russulaceae,Lactarius hatsudake,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,11,"x=0, e=500, c=1"
1699,Fungi,Mucoromycota,Mortierellomycetes,Mortierellales,Mortierellaceae,Mortierella hygrophila,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,5,"x=0, e=500, c=1"
685,Animalia,Chordata,Mammalia,Cingulata,Chlamyphoridae,Chaetophractus vellerosus,Complete,Dfam,NaN,Curated Dfam,Dfam,65,NaN
2209,Fungi,Basidiomycota,Agaricomycetes,Gomphales,Gomphaceae,Ramaria rubella,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,28,"x=0, e=500, c=1"
1551,Animalia,Chordata,Mammalia,Primates,Cercopithecidae,Macaca fascicularis,Complete,Dfam,NaN,Release 1.2,Dfam,56,NaN
2572,Fungi,Ascomycota,Sordariomycetes,Hypocreales,Ophiocordycipitaceae,Tolypocladium inflatum,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,1,"x=0, e=500, c=1"
1711,Fungi,Mucoromycota,Mucoromycetes,Mucorales,Mucoraceae,Mucor racemosus,Complete,RM2+MCH,NaN,"ENSEMBL ""2023""",RM2+MCH,24,"x=10, e=500, c=1"
2699,Fungi,Ascomycota,Dothideomycetes,Venturiales,Venturiaceae,Venturia nashicola,Complete,RM2+MCH,NaN,B-GUT,RM2+MCH,5,"x=0, e=500, c=1"


In [9]:
print(df.shape)# Devuelve una pareja de filas y columnas, sirve para saber el tamaño real de lo que se tiene
print(df.columns.tolist())#(df.columns)son los nombres de las columnas (.tolist()) los convierte en una lista de python, para verlo en una sola linea 


(2790, 13)
['Kingdom', 'Phylum', 'Class', 'Order', 'Family', 'Species', 'Library?', 'Source', 'Source 2', 'Notes', 'Selected', 'TE_count', 'MCH_Parameters']


In [7]:
#crea una tabla donde si esta vacio es true y los suma y si es no esta vacio o sea false 0
df.isna().sum()

Kingdom              0
Phylum               3
Class               49
Order               29
Family              76
Species              0
Library?             0
Source               0
Source 2          2740
Notes                6
Selected             0
TE_count             0
MCH_Parameters     386
dtype: int64

In [8]:
# df["Kingdom"] selecciona esa columna.
#.value_counts() cuenta cuántas veces aparece cada valor distinto.
#dropna=False hace que también cuente los vacíos. Sin eso, los NaN se ocultarían y no los verías.
df["Kingdom"].value_counts(dropna=False)

Kingdom
Fungi       1559
Animalia     780
Plantae      451
Name: count, dtype: int64

In [14]:
#.str.strip() quita los espacios al inicio y al final de cada nombre.
#!= compara el nombre original con el recortado: da True donde son distintos, o sea donde había espacios de más.
#.sum() cuenta los True.
(df["Species"] != df["Species"].str.strip()).sum()

np.int64(0)

In [11]:
#Compara cada nombre con su versión en minúsculas. Como todos los nombres de especie empiezan con mayúscula (Macaca fascicularis), todos difieren.
(df["Species"] != df["Species"].str.lower()).sum()

np.int64(2790)

In [12]:
#.str.lower() pasa todo a minúsculas.
#.duplicated() marca cada nombre que ya apareció antes.
#.sum() los cuenta.
df["Species"].str.lower().duplicated().sum()

np.int64(0)

In [13]:
#Cuenta cuántas especies hay por origen, incluyendo vacíos.
df["Source"].value_counts(dropna=False)

Source
RM2+MCH    2404
Dfam        386
Name: count, dtype: int64

In [17]:
df[["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]]
df_csv_9col = df[["Kingdom", "Phylum", "Class", "Order", "Family", "Species", "Source", "Source 2", "Notes"]].copy()

In [19]:
print(df_csv_9col.shape)
df_csv_9col.isna().sum()

(2790, 9)


Kingdom        0
Phylum         3
Class         49
Order         29
Family        76
Species        0
Source         0
Source 2    2740
Notes          6
dtype: int64

In [21]:
# Se normaliza Species con strip(): si el CSV y el FASTA difieren por un espacio sobrante, el cruce no los une
df_csv_9col["Species"] = df_csv_9col["Species"].str.strip() #Se Sobreescribe la columna con su version recortada

In [22]:
(df_csv_9col["Species"] != df_csv_9col["Species"].str.strip()).sum()

np.int64(0)

In [ ]:
#EXPLORACIÓN DE VACÍOS Y PROCEDENCIA

In [24]:
#Vacios
df_csv_9col[df_csv_9col["Phylum"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
1310,Fungi,NaN,Agaricomycetes,Agaricales,Hymenogastraceae,Hebeloma cylindrosporum,RM2+MCH,NaN,B-GUT
1617,Animalia,NaN,NaN,NaN,NaN,Metazoa,Dfam,Repbase,No especie Repbase
2143,Animalia,NaN,NaN,NaN,NaN,Protostomia,Dfam,NaN,Dfam no especie


In [25]:
df_csv_9col[df_csv_9col["Class"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
56,Fungi,Oomycota,NaN,Albuginales,Albuginaceae,Albugo candida,RM2+MCH,NaN,B-GUT
57,Fungi,Oomycota,NaN,Albuginales,Albuginaceae,Albugo laibachii,RM2+MCH,NaN,B-GUT
106,Plantae,Angiosperma,NaN,Amborellales,Amborellaceae,Amborella trichopoda,RM2+MCH,Repbase,Uncurated Dfam
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
170,Fungi,Oomycota,NaN,Saprolegniales,Saprolegniaceae,Aphanomyces astaci,RM2+MCH,NaN,B-GUT
204,Plantae,Angiosperma,NaN,Ranunculales,Ranunculaceae,Aquilegia eximia,RM2+MCH,NaN,Uncurated Dfam
484,Plantae,Angiosperma,NaN,Caryophyllales,Amaranthaceae,Beta vulgaris,RM2+MCH,Repbase,"ENSEMBL ""2023"""
520,Plantae,Angiosperma,NaN,Poales,Poaceae,Bonia amplexicaulis,RM2+MCH,NaN,Uncurated Dfam
547,Plantae,Angiosperma,NaN,Poales,Poaceae,Brachypodium distachyon,RM2+MCH,Repbase,Uncurated Dfam
583,Fungi,Mucoromycota,NaN,Calcarisporiellales,Calcarisporiellaceae,Calcarisporiella thermophila,RM2+MCH,NaN,B-GUT


In [26]:
df_csv_9col[df_csv_9col["Order"].isna()]

,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
509,Fungi,Chytridiomycota,Chytridiomycetes,NaN,NaN,Blyttiomyces helicus,RM2+MCH,NaN,B-GUT
830,Fungi,Ascomycota,Dothideomycetes,NaN,NaN,Coniosporium apollinis,RM2+MCH,NaN,B-GUT
873,Fungi,Ascomycota,Dothideomycetes,NaN,NaN,Cryomyces minteri,RM2+MCH,NaN,B-GUT
975,Fungi,Ascomycota,Dothideomycetes,NaN,Zopfiaceae,Didymocrea sadasivanii,RM2+MCH,NaN,B-GUT
1031,Fungi,Microsporidia,NaN,NaN,Culicosporidae,Edhazardia aedis,RM2+MCH,NaN,B-GUT
1129,Animalia,Chordata,NaN,NaN,NaN,Euteleostomi,Dfam,Repbase,No especie Repbase
1300,Fungi,Microsporidia,Dubosqiidae,NaN,NaN,Hamiltosporidium magnivora,RM2+MCH,NaN,B-GUT
1301,Fungi,Microsporidia,Dubosqiidae,NaN,NaN,Hamiltosporidium tvaerminnensis,RM2+MCH,NaN,B-GUT
1330,Fungi,Microsporidia,NaN,NaN,Hepatosporidae,Hepatospora eriocheir,RM2+MCH,NaN,B-GUT


In [27]:
df_csv_9col[df_csv_9col["Family"].isna()]


,Kingdom,Phylum,Class,Order,Family,Species,Source,Source 2,Notes
10,Fungi,Ascomycota,Sordariomycetes,Amplistromataceae,NaN,Acidothrix acidophila,RM2+MCH,NaN,B-GUT
39,Animalia,Chordata,Mammalia,No trobat,NaN,Afrotheria,Dfam,NaN,Dfam no especie
110,Animalia,Chordata,NaN,NaN,NaN,Amniota,Dfam,NaN,Dfam no especie
118,Fungi,Ascomycota,Leotiomycetes,Helotiales,NaN,Amylocarpus encephaloides,RM2+MCH,NaN,B-GUT
164,Fungi,Ascomycota,Sordariomycetes,Xylariales,NaN,Anthostomelloides leucospermi,RM2+MCH,NaN,B-GUT
...,...,...,...,...,...,...,...,...,...
2577,Fungi,Microsporidia,Pleistophoridae,NaN,NaN,Trachipleistophora hominis,RM2+MCH,NaN,B-GUT
2677,Fungi,Basidiomycota,Ustilaginomycetes,Ustilaginales,NaN,Ustilago maydis,RM2+MCH,NaN,B-GUT
2704,Fungi,Ascomycota,Dothideomycetes,Pleosporales,NaN,Verruculina enalia,RM2+MCH,NaN,B-GUT
2739,Animalia,Chordata,Mammalia,No trobat,NaN,Xenarthra,Dfam,NaN,Dfam no especie


In [28]:
df_csv_9col["Notes"].value_counts(dropna=False)

Notes
B-GUT                 1479
ENSEMBL "2023"         375
ENSEMBL "2025"         373
Curated Dfam           293
Uncurated Dfam         165
Release 1.2             43
Dfam no especie         34
APTEdb                  12
No especie Repbase      10
NaN                      6
Name: count, dtype: int64

In [29]:
df_csv_9col["Order"].value_counts(dropna=False).head(30)

Order
Eurotiales           267
Agaricales           127
Hypocreales          101
Pleosporales          78
Pezizales             73
Polyporales           66
Rodentia              64
Boletales             62
Sordariales           61
Poales                60
Mucorales             57
Helotiales            56
Primates              54
Artiodactyla          47
Glomerellales         46
Passeriformes         42
Russulales            38
Xylariales            36
Chiroptera            36
Myrtales              33
Carnivora             32
Lepidoptera           31
NaN                   29
Saccharomycetales     28
Malpighiales          26
Onygenales            24
Mortierellales        23
Fabales               23
Rosales               23
Lamiales              22
Name: count, dtype: int64

In [30]:
df_csv_9col[df_csv_9col["Class"].isna()]["Kingdom"].value_counts()

Kingdom
Fungi       21
Plantae     15
Animalia    13
Name: count, dtype: int64

In [31]:
df_csv_9col["Order"].value_counts(dropna=False).tail(15)

Order
Thysanoptera                               1
Hypnales                                   1
Scombriformes                              1
Harpacticoida                              1
Trichinellida                              1
Phaeotrichales                             1
Celastrales                                1
Trogoniformes                              1
Ulvales                                    1
Umbelopsidales                             1
Hirudinida                                 1
Teloschistales                             1
Pandanales                                 1
Xylonales                                  1
No order Dothideomycetes incertae sedis    1
Name: count, dtype: int64

In [32]:
marcadores = ["No trobat", "No order"]
df_csv_9col.isin(marcadores).sum()

Kingdom      0
Phylum       0
Class        0
Order       12
Family       0
Species      0
Source       0
Source 2     0
Notes        0
dtype: int64

In [41]:
df_csv_9col[df_csv_9col["Order"].str.startswith("No ", na=False)]["Order"].value_counts()

Order
No trobat                                       7
No order                                        5
No ordre NCBI                                   2
No Orden No trobat                              2
No ordre NCBI Caenogastropoda incertae sedis    1
No order (eupecaria incertae sedis)             1
No order Dothideomycetes incertae sedis         1
Name: count, dtype: int64

In [42]:
df_csv_9col["Order"].isna().sum()

np.int64(29)

In [49]:
df_csv_9col.loc[df_csv_9col["Order"].str.startswith("No ", na=False) , "Order" ] = np.nan

In [50]:
print(df_csv_9col["Order"].isna().sum())
print(df_csv_9col.shape)

48
(2790, 9)


In [34]:
df_csv_9col[df_csv_9col["Class"].str.startswith("No ", na=False)]["Class"].value_counts()

Series([], Name: count, dtype: int64)

In [38]:
df_csv_9col[df_csv_9col["Family"].str.startswith("Unknown ", na=False)]["Family"].value_counts()

Series([], Name: count, dtype: int64)

In [1]:
#ARCHIVO FASTA
from Bio import SeqIO
ruta_fasta = "../data_raw/PanTEon_Database_v1.6.1.fasta"


In [5]:
for i, registro in enumerate( SeqIO.parse(ruta_fasta, "fasta")):
    #registro.id: lo que hay en el encabezado después del > y hasta el primer espacio.
    print(registro.id)
    #registro.description: el encabezado completo, sin el >.
    print(registro.description)
    #registro.seq: la secuencia de nucleótidos. Es un objeto Seq, que se comporta parecido a un texto.
    print(len(registro.seq))
    if i == 4:
        break

PDB00000001#CLASSII/HELITRON/HELITRON
PDB00000001#CLASSII/HELITRON/HELITRON @Halicephalobus mephisto
1991
PDB00000002#CLASSII/HELITRON/HELITRON
PDB00000002#CLASSII/HELITRON/HELITRON @Halicephalobus mephisto
1689
PumCon-1.11#CLASSI/SINE/TRNA
PumCon-1.11#CLASSI/SINE/TRNA @Puma concolor
235
PumCon-1.141#CLASSII/TIR/TC1MARINER
PumCon-1.141#CLASSII/TIR/TC1MARINER @Puma concolor
340
PumCon-1.14#CLASSI/SINE/TRNA
PumCon-1.14#CLASSI/SINE/TRNA @Puma concolor
233


In [13]:
#Separacion de la cadena del ID de la clasificacion, la clasificacion de cada especie y la clase, orden y superfamilia entre si
def separar_cadena(cadena):
    #split("@", 1): corta en el primer @ y da dos pedazos (izquierda y especie). El 1 limita a un solo corte.
    izquierda, especie = cadena.split("@", 1)
    #split("#", 1): igual, separa ID y clasificación.
    id_seq, clasificacion = izquierda.split("#", 1)
    #split("/"): separa en tres niveles. Si algún encabezado no tiene exactamente tres, da error; eso lo manejamos en el paso 3.
    clase_fasta, orden_fasta, superfamilia_fasta = clasificacion.strip().split("/")
    #strip() quita espacios sobrantes en cada pedazo.
    return (id_seq.strip(), clase_fasta.strip(), orden_fasta.strip(), superfamilia_fasta.strip(), especie.strip())

In [14]:
for i, registro in enumerate(SeqIO.parse("../data_raw/PanTEon_Database_v1.6.1.fasta", "fasta")):
    print(separar_cadena(registro.description))
    if i == 4:
        break

('PDB00000001', 'CLASSII', 'HELITRON', 'HELITRON', 'Halicephalobus mephisto')
('PDB00000002', 'CLASSII', 'HELITRON', 'HELITRON', 'Halicephalobus mephisto')
('PumCon-1.11', 'CLASSI', 'SINE', 'TRNA', 'Puma concolor')
('PumCon-1.141', 'CLASSII', 'TIR', 'TC1MARINER', 'Puma concolor')
('PumCon-1.14', 'CLASSI', 'SINE', 'TRNA', 'Puma concolor')
